# 🛒 Retail Basket Value Prediction - EDA & Feature Engineering

**Dataset:** Online Retail Dataset  
**Author:** Kalash Mishra  
**Goal:** Predict customer basket value

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

print('✅ Libraries imported!')

In [ ]:
df = pd.read_csv('Data/Online Retail.csv', encoding='latin1')
print(f'Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
print('Missing Values:')
print(missing_df[missing_df['Count'] > 0])

In [ ]:
plt.figure(figsize=(10, 6))
missing_data = missing_df[missing_df['Count'] > 0]
if len(missing_data) > 0:
    plt.barh(missing_data.index, missing_data['Percentage'], color='coral')
    plt.xlabel('Missing %')
    plt.title('Missing Values by Column')
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
print('Top 10 Countries:')
top_countries = df['Country'].value_counts().head(10)
print(top_countries)

plt.figure(figsize=(12, 6))
top_countries.plot(kind='barh', color='skyblue')
plt.xlabel('Transactions')
plt.title('Top 10 Countries')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
print('Quantity Statistics:')
print(f"Mean: {df['Quantity'].mean():.2f}")
print(f"Median: {df['Quantity'].median():.2f}")
print(f"Min: {df['Quantity'].min()}")
print(f"Max: {df['Quantity'].max()}")

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
positive_qty = df[df['Quantity'] > 0]['Quantity']
plt.hist(positive_qty[positive_qty <= 50], bins=50, color='lightgreen', edgecolor='black')
plt.xlabel('Quantity')
plt.ylabel('Frequency')
plt.title('Quantity Distribution (1-50)')

plt.subplot(1, 2, 2)
plt.boxplot(positive_qty[positive_qty <= 100])
plt.ylabel('Quantity')
plt.title('Quantity Boxplot')
plt.tight_layout()
plt.show()

## Data Cleaning

In [ ]:
df_clean = df.copy()
print(f'Initial: {len(df_clean):,} rows')

# Remove missing CustomerID
df_clean = df_clean.dropna(subset=['CustomerID'])
print(f'After removing missing CustomerID: {len(df_clean):,}')

# Remove cancelled transactions
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
print(f'After removing cancellations: {len(df_clean):,}')

# Remove invalid quantities and prices
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
print(f'After removing invalid values: {len(df_clean):,}')

# Remove duplicates
df_clean = df_clean.drop_duplicates()
print(f'Final: {len(df_clean):,} rows')

## Feature Engineering

In [ ]:
# Convert date
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Extract time features
df_clean['hour'] = df_clean['InvoiceDate'].dt.hour
df_clean['day_of_week'] = df_clean['InvoiceDate'].dt.dayofweek
df_clean['month'] = df_clean['InvoiceDate'].dt.month
df_clean['is_weekend'] = (df_clean['day_of_week'] >= 5).astype(int)
df_clean['is_peak_hour'] = ((df_clean['hour'] >= 10) & (df_clean['hour'] <= 15)).astype(int)

# Calculate total price
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

print('✅ Time features created')

In [ ]:
invoice_features = df_clean.groupby('InvoiceNo').agg({
    'TotalPrice': 'sum',
    'Quantity': 'sum',
    'StockCode': 'count',
    'CustomerID': 'first',
    'Country': 'first',
    'InvoiceDate': 'first',
    'hour': 'first',
    'day_of_week': 'first',
    'month': 'first',
    'is_weekend': 'first',
    'is_peak_hour': 'first'
}).reset_index()

invoice_features.rename(columns={
    'TotalPrice': 'basket_value',
    'Quantity': 'total_quantity',
    'StockCode': 'num_items'
}, inplace=True)

print(f'✅ {len(invoice_features):,} invoices aggregated')

In [ ]:
customer_stats = invoice_features.groupby('CustomerID').agg({
    'basket_value': ['mean', 'sum', 'count'],
    'total_quantity': 'mean',
    'num_items': 'mean'
}).reset_index()

customer_stats.columns = ['CustomerID', 'avg_basket_value', 'total_spent', 
                          'purchase_count', 'avg_quantity', 'avg_unique_items']

invoice_features = invoice_features.merge(customer_stats, on='CustomerID', how='left')

# Derived features
invoice_features['avg_item_price'] = invoice_features['basket_value'] / invoice_features['total_quantity']
invoice_features['item_diversity'] = invoice_features['num_items'] / invoice_features['total_quantity']

# Loyalty score
max_purchases = invoice_features['purchase_count'].max()
invoice_features['customer_loyalty'] = invoice_features['purchase_count'] / max_purchases

# Location features
invoice_features['is_uk'] = (invoice_features['Country'] == 'United Kingdom').astype(int)
le = LabelEncoder()
invoice_features['country_encoded'] = le.fit_transform(invoice_features['Country'])

# Recency
max_date = invoice_features['InvoiceDate'].max()
invoice_features['recency_days'] = (max_date - invoice_features['InvoiceDate']).dt.days

print('✅ All features created')

In [ ]:
numerical_features = invoice_features.select_dtypes(include=[np.number]).columns.tolist()
correlation_matrix = invoice_features[numerical_features].corr()

plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

print('\nTop correlations with basket_value:')
print(correlation_matrix['basket_value'].sort_values(ascending=False)[1:6])

In [ ]:
import os
output_file = 'Data/processed_retail_data.csv'
invoice_features.to_csv(output_file, index=False)

print(f'✅ Saved to: {output_file}')
print(f'Records: {len(invoice_features):,}')
print(f'Features: {len(invoice_features.columns)}')
print(f'Size: {os.path.getsize(output_file) / (1024*1024):.2f} MB')

In [ ]:
print('='*70)
print('FINAL DATASET SUMMARY')
print('='*70)
print(f'Total Records: {len(invoice_features):,}')
print(f'Total Features: {len(invoice_features.columns)}')
print('\nFeatures for Model Training:')
feature_cols = ['total_quantity', 'num_items', 'hour', 'day_of_week', 'month',
                'is_weekend', 'is_peak_hour', 'avg_basket_value', 'total_spent',
                'purchase_count', 'avg_quantity', 'avg_unique_items', 'avg_item_price',
                'item_diversity', 'customer_loyalty', 'is_uk', 'country_encoded', 'recency_days']
for i, f in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {f}')
print(f'\nTarget: basket_value')
print('='*70)